# 05 Calibration

This notebook fits a simple gravity model to real attendance of attractions along the Klint.

**Model.** Visitors to attraction j = k × Σ over origins i of P_i × exp(-β × t_ij), where P_i is the population of origin i and t_ij the car travel time in minutes (notebook 04). β is the distance decay: how quickly interest falls with travel time. k is an overall scale.

**Why only k and β.** We have 8 attendance figures. Giving each attraction its own attractiveness would fit the data perfectly and predict nothing. Instead, the ratio observed / predicted for each site is read as its **attractiveness factor**, which notebook 06 uses to build low/base/high scenarios for the new museum.

**Attendance data.** `data/attendance_by_year.csv`: official figures from GBUK LO "Museum Agency" (letter of 12.11.2026, publication allowed with credit) for Slantsy, Kingisepp, Ivangorod, Koporye and Izvara; Putilovo from the park's reply; Staraya Ladoga (2021) and Oreshek (2023) from press reports. Years with a flag (closed for repairs, 2026 petrol crisis) are excluded, and the latest remaining year is used. Sablino is not used: cave tours are capped and most visitors cannot be counted.

**Fit.** In log space (each site counts equally, whether it has 5,000 or 200,000 visitors), with scipy `least_squares`. Checked with leave-one-out.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

PROCESSED = "../data/processed"

origins = pd.read_csv(f"{PROCESSED}/origins.csv")
tt = pd.read_csv(f"{PROCESSED}/travel_times.csv").set_index("origin_id")
att = pd.read_csv("../data/attractions.csv")
years = pd.read_csv("../data/attendance_by_year.csv")

# population of the 3,440 origins that have travel times (inner join)
pop = origins.set_index("origin_id")["population"].reindex(tt.index)
assert pop.notna().all()
print(f"Origins: {len(tt):,}, population {pop.sum():,.0f}")

# names must match across files
assert set(att["name_en"]) == set(tt.columns)
assert set(years["name_en"]) <= set(att["name_en"]), set(years["name_en"]) - set(att["name_en"])

# reference value: latest year without a flag
ok = years[years["flag"].isna()]
ref = ok.sort_values("year").groupby("name_en").tail(1).set_index("name_en")
avg = ok[ok["year"].between(2023, 2025)].groupby("name_en")["visitors"].mean()  # for a sensitivity check

calib = (att.loc[att["role"] == "calibration", ["name_en", "type"]]
            .set_index("name_en")
            .join(ref[["year", "visitors"]], how="inner"))
calib["avg_23_25"] = avg.round(0)
calib = calib.sort_values("visitors", ascending=False)
print(f"Calibration points: {len(calib)}")
calib

Origins: 3,440, population 8,265,011
Calibration points: 8


,type,year,visitors,avg_23_25
name_en,,,,
Oreshek Fortress (Shlisselburg),fortress,2023,198000,198000.0
Staraya Ladoga Museum-Reserve,museum-reserve,2021,140000,NaN
Koporye Fortress,fortress,2025,47179,42444.0
Ivangorod Museum (fortress and galleries),fortress / museum,2025,40656,39194.0
Putilovo Paleopark (fossil digging),geology park,2025,13000,11667.0
Roerich Estate Museum (Izvara),museum,2025,10713,10148.0
Kingisepp Local History Museum,museum,2025,9377,8704.0
Slantsy Local History Museum,museum,2025,4645,4645.0
